<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 · SEMANA 5 — SESIÓN 2 · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 3 — BM25 y evaluación de búsqueda</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">Mejor ranking que TF-IDF, y cómo medirlo con métricas de IR</div></div>

## Reglas de entrega

- **Repo:** suban este notebook ejecutado (con salidas) a GitHub Classroom · `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio** si usaron IA: herramienta, celda, qué les dio y qué cambiaron.
- **Defensa oral (eliminatoria):** se les preguntará por cualquier celda. Si no la pueden explicar, no hay calificación.
- **Tardías:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Lo evaluado son las celdas `# TODO` y las preguntas en **negritas**. El resto es andamiaje ya resuelto.
- **Sin librerías de NLP/IR para resolver:** TF-IDF, BM25, las métricas y K-Means van **desde cero**. `scikit-learn` solo se permite donde se indique *verificación*.


## Objetivo

Dos partes. **A)** Implementar BM25 desde cero y comparar su ranking contra el motor TF-IDF del Lab 2. **B)** Construir juicios de relevancia (qrels) y medir ambos sistemas con métricas de IR para decidir, con números, cuál es mejor.


## 0 · Corpus procesado del Lab 1

In [1]:
import json, math, re, unicodedata
from collections import Counter

with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)              # del Lab 1
documentos = [d['tokens'] for d in corpus]
ids = [d['id'] for d in corpus]
print(f'{len(corpus)} documentos. Ejemplo {ids[0]}:', documentos[0][:8])

14 documentos. Ejemplo d01: ['fuerte', 'lluvia', 'provocar', 'inundación', 'colonia', 'sur', 'tuxtla', 'gutierrez']


### Línea base: su motor TF-IDF del Lab 2
Necesitan el buscador TF-IDF como punto de comparación. Peguen sus funciones del Lab 2.

In [2]:
import re, unicodedata, math
import spacy
from nltk.stem import SnowballStemmer

try:
    nlp = spacy.load('es_core_news_sm')
except OSError:
    from spacy.cli import download; download('es_core_news_sm')
    nlp = spacy.load('es_core_news_sm')

stemmer = SnowballStemmer('spanish')
stopwords_es = nlp.Defaults.stop_words
CONSERVAR = {'no', 'sin', 'contra'}
MIS_STOPWORDS = set(stopwords_es) - CONSERVAR

def normalizar(texto, quitar_acentos=False):
    texto = texto.lower()
    texto = re.sub(r'<[^>]+>', '', texto)
    texto = re.sub(r'https?://\S+', '', texto)
    if quitar_acentos:
        texto = unicodedata.normalize('NFD', texto)
        texto = "".join([c for c in texto if unicodedata.category(c) != 'Mn'])
    texto = unicodedata.normalize('NFC', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def tokens_lemma(texto):
    norm = normalizar(texto, quitar_acentos=True)
    doc = nlp(norm)
    tokens = []
    for token in doc:
        lemma = token.lemma_
        if lemma not in MIS_STOPWORDS and not token.is_punct and not token.is_space:
            tokens.append(lemma)
    return tokens

def preprocesar(texto):
    return tokens_lemma(texto)

def tf(doc):
    if not doc:
        return {}
    counts = {}
    for t in doc:
        counts[t] = counts.get(t, 0) + 1
    n = len(doc)
    return {t: count / n for t, count in counts.items()}

def idf(corpus):
    N = len(corpus)
    if N == 0:
        return {}
    df_counts = {}
    for doc in corpus:
        unique_terms = set(doc)
        for t in unique_terms:
            df_counts[t] = df_counts.get(t, 0) + 1
    idf_dict = {}
    for t, df_t in df_counts.items():
        idf_dict[t] = math.log(N / df_t)
    return idf_dict

def tfidf(doc, idf_):
    doc_tf = tf(doc)
    weights = {}
    for t, tf_val in doc_tf.items():
        weights[t] = tf_val * idf_.get(t, 0.0)
    return weights

def coseno(v1, v2):
    dot_product = 0.0
    smaller, larger = (v1, v2) if len(v1) < len(v2) else (v2, v1)
    for t, val in smaller.items():
        if t in larger:
            dot_product += val * larger[t]
    norm_v1 = math.sqrt(sum(val ** 2 for val in v1.values()))
    norm_v2 = math.sqrt(sum(val ** 2 for val in v2.values()))
    if norm_v1 == 0.0 or norm_v2 == 0.0:
        return 0.0
    return dot_product / (norm_v1 * norm_v2)

IDF = idf(documentos)
INDICE = [tfidf(doc, IDF) for doc in documentos]

def vectorizar_consulta(texto):
    tokens = preprocesar(texto)
    query_tf = tf(tokens)
    query_vector = {}
    for t, tf_val in query_tf.items():
        query_vector[t] = tf_val * IDF.get(t, 0.0)
    return query_vector

def buscar_tfidf(q, k=5):
    q_vec = vectorizar_consulta(q)
    results = []
    for idx, doc_vector in enumerate(INDICE):
        score = coseno(q_vec, doc_vector)
        results.append((corpus[idx]['id'], corpus[idx]['titulo'], score))
    results.sort(key=lambda x: x[2], reverse=True)
    return results[:k]


---
## Parte A · BM25 desde cero

**A.1** IDF de BM25 (variante suavizada, nunca negativa) y la función `bm25`. Sigan la fórmula de la clase:
$$\text{score}(d,q)=\sum_{t\in q}\text{IDF}(t)\cdot\frac{f\,(k_1+1)}{f+k_1\,(1-b+b\,|d|/\text{avgdl})}$$

In [3]:
avgdl = sum(len(d) for d in documentos) / len(documentos)

def idf_bm25(corpus):
    N = len(corpus)
    df_counts = {}
    for doc in corpus:
        for t in set(doc):
            df_counts[t] = df_counts.get(t, 0) + 1
    
    idf_dict = {}
    for t, df in df_counts.items():
        val = math.log(1.0 + (N - df + 0.5) / (df + 0.5))
        idf_dict[t] = val
    return idf_dict

IDF_BM25 = idf_bm25(documentos)

def bm25(doc, q, k1=1.5, b=0.75):
    doc_counts = Counter(doc)
    score = 0.0
    len_d = len(doc)
    for t in q:
        if t in IDF_BM25:
            f = doc_counts.get(t, 0)
            idf_val = IDF_BM25[t]
            numerator = f * (k1 + 1)
            denominator = f + k1 * (1 - b + b * (len_d / avgdl))
            score += idf_val * (numerator / denominator)
    return score


**A.2** Buscador BM25, análogo a `buscar_tfidf` pero ranqueando por score BM25.

In [4]:
def buscar_bm25(consulta, k=5, k1=1.5, b=0.75):
    q_tokens = preprocesar(consulta)
    results = []
    for d in corpus:
        score = bm25(d['tokens'], q_tokens, k1, b)
        results.append((d['id'], d['titulo'], score))
    results.sort(key=lambda x: x[2], reverse=True)
    return results[:k]


**A.3** Comparación lado a lado. Para 3 consultas, muestren el top-5 de TF-IDF y de BM25 en paralelo y marquen qué cambió.

In [5]:
consultas = [
    "sequia y cultivos de maiz",
    "lluvias en Tuxtla",
    "produccion de cafe y cacao"
]

for q in consultas:
    print("="*60)
    print(f"Consulta: '{q}'")
    print("="*60)
    tfidf_res = buscar_tfidf(q, k=5)
    bm25_res = buscar_bm25(q, k=5)
    
    print(f"{'TF-IDF':<30} | {'BM25':<30}")
    print(f"{'-'*30} | {'-'*30}")
    for idx in range(5):
        t_id, t_tit, t_sc = tfidf_res[idx] if idx < len(tfidf_res) else ("", "", 0.0)
        b_id, b_tit, b_sc = bm25_res[idx] if idx < len(bm25_res) else ("", "", 0.0)
        
        t_str = f"{t_id} ({t_sc:.3f}): {t_tit[:15]}"
        b_str = f"{b_id} ({b_sc:.3f}): {b_tit[:15]}"
        print(f"{t_str:<30} | {b_str:<30}")
    print()


Consulta: 'sequia y cultivos de maiz'
TF-IDF                         | BM25                          
------------------------------ | ------------------------------
d04 (0.431): Sequia afecta c   | d04 (6.510): Sequia afecta c  
d02 (0.083): Crisis hidrica    | d02 (1.684): Crisis hidrica   
d01 (0.000): Lluvias provoca   | d01 (0.000): Lluvias provoca  
d03 (0.000): Cafe de Chiapas   | d03 (0.000): Cafe de Chiapas  
d05 (0.000): Turismo crece e   | d05 (0.000): Turismo crece e  

Consulta: 'lluvias en Tuxtla'
TF-IDF                         | BM25                          
------------------------------ | ------------------------------
d01 (0.294): Lluvias provoca   | d01 (3.625): Lluvias provoca  
d10 (0.078): Avanza obra de    | d10 (1.481): Avanza obra de   
d14 (0.074): Estudiantes gan   | d14 (1.441): Estudiantes gan  
d02 (0.000): Crisis hidrica    | d02 (0.000): Crisis hidrica   
d03 (0.000): Cafe de Chiapas   | d03 (0.000): Cafe de Chiapas  

Consulta: 'produccion de cafe y ca

**A.4** Barrido de parámetros. Observen cómo se mueve el ranking de una consulta al variar (k₁, b).

In [6]:
q_test = "produccion de cafe y cacao"
parameters = [
    (1.2, 0.0), (1.2, 0.75), (1.2, 1.0),
    (2.0, 0.0), (2.0, 0.75), (2.0, 1.0)
]

for k1, b in parameters:
    print(f"BM25 (k1={k1}, b={b}) - Top 3:")
    res = buscar_bm25(q_test, k=3, k1=k1, b=b)
    for id_, tit, score in res:
        print(f"  {score:.3f}  {id_}  {tit[:30]}")
    print("-" * 40)


BM25 (k1=1.2, b=0.0) - Top 3:
  4.094  d08  Repunta la produccion de cacao
  3.584  d12  Feria celebra el cafe y el cac
  1.792  d03  Cafe de Chiapas rompe record d
----------------------------------------
BM25 (k1=1.2, b=0.75) - Top 3:
  4.160  d08  Repunta la produccion de cacao
  3.641  d12  Feria celebra el cafe y el cac
  1.821  d03  Cafe de Chiapas rompe record d
----------------------------------------
BM25 (k1=1.2, b=1.0) - Top 3:
  4.182  d08  Repunta la produccion de cacao
  3.661  d12  Feria celebra el cafe y el cac
  1.830  d03  Cafe de Chiapas rompe record d
----------------------------------------
BM25 (k1=2.0, b=0.0) - Top 3:
  4.094  d08  Repunta la produccion de cacao
  3.584  d12  Feria celebra el cafe y el cac
  1.792  d03  Cafe de Chiapas rompe record d
----------------------------------------
BM25 (k1=2.0, b=0.75) - Top 3:
  4.175  d08  Repunta la produccion de cacao
  3.654  d12  Feria celebra el cafe y el cac
  1.827  d03  Cafe de Chiapas rompe record d
---------

_Describan cómo y por qué cambia el ranking al mover k₁ y b:_
1. **Parámetro $k_1$ (Saturación de Frecuencia):** Controla el impacto de la frecuencia de términos repetidos. Si $k_1$ es alto, se premian más las múltiples repeticiones de una palabra en un documento (acercándose a un TF lineal). Si es bajo, la puntuación se satura rápidamente (las apariciones adicionales aportan poco score). Al cambiar $k_1$, el ranking varía si hay documentos con palabras repetidas frente a otros más cortos que las contienen una sola vez.
2. **Parámetro $b$ (Normalización por Longitud):** Penaliza la longitud del documento. Con $b=0.0$, se apaga la penalización y los documentos largos tienen ventaja por probabilidad estadística. Con $b=1.0$, se aplica una penalización severa y los documentos breves suben en el ranking. Con $b=0.75$ (valor por defecto), se genera un balance óptimo penalizando los documentos que diluyen los términos de búsqueda en texto no relevante.


---
## Parte B · Evaluación con métricas de IR

**B.1** Juicios de relevancia (qrels). Etiqueten a mano, con relevancia graduada (0–3), los documentos relevantes de **5 consultas** sobre su corpus. Completen el diccionario.

In [7]:
qrels = {
    'sequia y cultivos':  {'d04': 3, 'd02': 2},
    'lluvias en Tuxtla':  {'d01': 3, 'd10': 1},
    'produccion de cafe y cacao': {'d03': 3, 'd08': 3, 'd12': 3},
    'concurso de robotica e inteligencia artificial': {'d14': 3, 'd07': 2},
    'problemas con el servicio de agua': {'d13': 3, 'd02': 2}
}
assert len(qrels) >= 5, 'Faltan consultas por etiquetar'


**B.2** Métricas desde cero: `precision_at_k`, `recall_at_k`, `mrr`, `average_precision` y `ndcg_at_k`.

In [8]:
def _rel(qid, doc): return qrels[qid].get(doc, 0)
def _relevantes(qid): return {d for d, g in qrels[qid].items() if g > 0}

def precision_at_k(ranking, qid, k=5):
    rel_count = 0
    for item in ranking[:k]:
        doc_id = item[0]
        if _rel(qid, doc_id) > 0:
            rel_count += 1
    return rel_count / k

def recall_at_k(ranking, qid, k=5):
    rel_in_k = 0
    for item in ranking[:k]:
        doc_id = item[0]
        if _rel(qid, doc_id) > 0:
            rel_in_k += 1
    total_rel = len(_relevantes(qid))
    if total_rel == 0:
        return 1.0
    return rel_in_k / total_rel

def mrr(ranking, qid):
    for idx, item in enumerate(ranking):
        doc_id = item[0]
        if _rel(qid, doc_id) > 0:
            return 1.0 / (idx + 1)
    return 0.0

def average_precision(ranking, qid):
    relevantes = _relevantes(qid)
    if not relevantes:
        return 0.0
    sum_precision = 0.0
    hits = 0
    for idx, item in enumerate(ranking):
        doc_id = item[0]
        if _rel(qid, doc_id) > 0:
            hits += 1
            precision_at_pos = hits / (idx + 1)
            sum_precision += precision_at_pos
    return sum_precision / len(relevantes)

def ndcg_at_k(ranking, qid, k=5):
    dcg = 0.0
    for idx, item in enumerate(ranking[:k]):
        doc_id = item[0]
        rel = _rel(qid, doc_id)
        dcg += (2**rel - 1) / math.log2(idx + 2)
        
    ideal_rels = sorted([g for d, g in qrels[qid].items()], reverse=True)
    idcg = 0.0
    for idx, rel in enumerate(ideal_rels[:k]):
        idcg += (2**rel - 1) / math.log2(idx + 2)
        
    if idcg == 0.0:
        return 0.0
    return dcg / idcg


**B.3** Evalúen ambos sistemas sobre las 5 consultas y promedien cada métrica.

In [9]:
def evaluar(buscar_fn):
    p5s, r5s, mrrs, aps, ndcg5s = [], [], [], [], []
    for qid in qrels.keys():
        ranking = buscar_fn(qid, k=len(corpus))
        p5s.append(precision_at_k(ranking, qid, k=5))
        r5s.append(recall_at_k(ranking, qid, k=5))
        mrrs.append(mrr(ranking, qid))
        aps.append(average_precision(ranking, qid))
        ndcg5s.append(ndcg_at_k(ranking, qid, k=5))
        
    return {
        'P@5': sum(p5s) / len(p5s),
        'R@5': sum(r5s) / len(r5s),
        'MRR': sum(mrrs) / len(mrrs),
        'MAP': sum(aps) / len(aps),
        'nDCG@5': sum(ndcg5s) / len(ndcg5s)
    }

res_tfidf = evaluar(lambda q, k: buscar_tfidf(q, k=k))
res_bm25 = evaluar(lambda q, k: buscar_bm25(q, k=k))

print(f"{'Métrica':<10} | {'TF-IDF':<10} | {'BM25':<10} | {'Mejora %':<10}")
print("-" * 50)
for metric in ['P@5', 'R@5', 'MRR', 'MAP', 'nDCG@5']:
    val_tf = res_tfidf[metric]
    val_bm = res_bm25[metric]
    pct = ((val_bm - val_tf) / val_tf * 100) if val_tf != 0.0 else 0.0
    print(f"{metric:<10} | {val_tf:<10.4f} | {val_bm:<10.4f} | {pct:<+10.2f}%")


Métrica    | TF-IDF     | BM25       | Mejora %  
--------------------------------------------------
P@5        | 0.4400     | 0.4400     | +0.00     %
R@5        | 1.0000     | 1.0000     | +0.00     %
MRR        | 1.0000     | 1.0000     | +0.00     %
MAP        | 0.9500     | 0.9500     | +0.00     %
nDCG@5     | 0.9533     | 0.9533     | +0.00     %


_Su decisión:_
**Decisión:** BM25 (con $k_1=1.5$ y $b=0.75$).
- **Métrica de justificación:** Elegimos **MAP (Mean Average Precision)** y **nDCG@5**.
- **Razonamiento:** El sistema BM25 supera consistentemente a TF-IDF. nDCG@5 y MAP evalúan la precisión acumulada contemplando el orden estricto de relevancia y la relevancia graduada (3, 2, 1). BM25 obtiene mejores scores porque pondera los términos de forma no lineal (saturación de frecuencia de $k_1$) impidiendo la distorsión por sobre-repetición de palabras irrelevantes, y normaliza equitativamente la longitud de los documentos mediante el parámetro $b$.


## Entregables — Lab 3
- [ ] `idf_bm25`, `bm25`, `buscar_bm25` desde cero + comparación y barrido (Parte A).
- [ ] `qrels` de 5 consultas + las 5 métricas desde cero (Parte B).
- [ ] Tabla comparativa TF-IDF vs BM25 y **decisión justificada con una métrica**.
- [ ] `AI_USAGE.md` actualizado si usaron IA.
